# Chat Anonymizer

Anonimizza i file di chat in `Chats/` usando Qwen3-32B, un chunk alla volta.

**Sostituzioni applicate:**
- Nomi reali di persone → `Persona`
- Nomi di aziende / clienti → `Azienda`
- Codici identificativi (pallet ID, LU-ID, dialog ref tipo MF12345, ecc.) → `COD`
- Prezzi e importi monetari → `PREZZO`
- `wamas` / `WAMAS` → `wms` (post-processing regex)

Ogni chunk è anonimizzato in isolamento, senza mantenere una mappa nome→etichetta:
l'anonimizzazione è quindi irreversibile (non esiste da nessuna parte una corrispondenza
salvata tra il testo originale e le etichette).

**Output:** file con suffisso `_anon.txt` nella stessa cartella dell'input.

In [ ]:
from llama_index.llms.openai_like import OpenAILike
from llama_index.core import PromptTemplate
import os, re
from dotenv import load_dotenv

load_dotenv()

llm = OpenAILike(
    model=os.getenv("MODEL_NAME"),  # deve corrispondere al nome caricato in LM Studio
    api_base = os.getenv("API_BASE"),
    api_key="lm-studio",  # LM Studio ignora il valore, ma OpenAILike vuole una stringa non vuota
    is_chat_model=True,
    is_function_calling_model=True,
    timeout=120.0,
    temperature=0.1,
    context_window=12288,
)

print("Setup completato.")

In [ ]:
response = llm.complete("ciao")
print(response.text)

In [ ]:
def load_lines(filepath: str) -> list:
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f if line.strip()]


def apply_wamas(text: str) -> str:
    return re.sub(r"\bwamas\b", "wms", text, flags=re.IGNORECASE)


def strip_code_fence(text: str) -> str:
    """Rimuove eventuali ``` che il modello a volte aggiunge attorno al testo."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n", "", text)
        text = re.sub(r"\n?```$", "", text)
    return text


print("Helper pronti.")

In [ ]:
ANON_PROMPT = PromptTemplate(
    "Sei un assistente che anonimizza trascrizioni di chat di supporto tecnico per sistemi warehouse.\n\n"
    "TESTO DA ANONIMIZZARE:\n"
    "{chunk}\n\n"
    "ISTRUZIONI:\n"
    "1. Sostituisci:\n"
    "   - Nomi reali di persone (nome, cognome o entrambi) → Persona\n"
    "   - Nomi di aziende o clienti reali → Azienda\n"
    "   - Codici identificativi: ID numerici lunghi, pallet ID, LU-ID, "
    "     riferimenti dialogo (MF12345, CU384, ecc.) → COD\n"
    "   - Prezzi e importi monetari → PREZZO\n"
    "2. Non serve mantenere coerenza tra un'occorrenza e l'altra: ogni entità va "
    "   sostituita con l'etichetta generica corrispondente, senza numerazione.\n"
    "3. NON sostituire:\n"
    "   - Timestamp (date e ore)\n"
    "   - Termini tecnici: AGV, WMS, OIL, JIRA, ITS, TPO, QC, AF, ecc.\n"
    "   - Nomi di file allegati (es. IMG-20210624-WA0000.jpg)\n"
    "   - Emoji, simboli speciali\n"
    "4. Mantieni la struttura esatta del testo (righe, spaziatura, punteggiatura).\n"
    "5. Restituisci ESCLUSIVAMENTE il testo anonimizzato, senza commenti, spiegazioni "
    "   o blocchi di codice attorno."
)

print("Prompt pronto.")

In [ ]:
def anonymize_file(filepath: str, output_path: str, window_size: int = 50):
    lines = load_lines(filepath)
    total = len(lines)
    base = os.path.basename(filepath)
    print(f"[{base}] {total} righe caricate")

    anonymized_lines: list = []
    current_idx = 0

    while current_idx < total:
        end_idx = min(current_idx + window_size, total)
        chunk = "\n".join(lines[current_idx:end_idx])

        print(f"  Righe [{current_idx+1}–{end_idx}] ...", end=" ", flush=True)

        try:
            response = llm.complete(ANON_PROMPT.format(chunk=chunk))
            anonymized_text = strip_code_fence(response.text)
            print("ok")
        except Exception as e:
            print(f"ERRORE LLM: {e}")
            anonymized_text = chunk

        anonymized_lines.extend(anonymized_text.split("\n"))
        current_idx = end_idx

    final_text = "\n".join(anonymized_lines)
    final_text = apply_wamas(final_text)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(final_text)

    print(f"\n  → Salvato in: {output_path}")


print("Funzione pronta.")

In [ ]:
INPUT_FILE  = "../Chats/ChatSSI_clean.txt"
OUTPUT_FILE = "../Chats/ChatSSI_clean_anon.txt"
WINDOW_SIZE = 50

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}\n")

anonymize_file(INPUT_FILE, OUTPUT_FILE, window_size=WINDOW_SIZE)